In [1]:
# import libraries
try:
  # %tensorflow_version only exists in Colab.
  !pip install tf-nightly
except Exception:
  pass
import tensorflow as tf
import pandas as pd
from tensorflow import keras
!pip install tensorflow-datasets
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt

print(tf.__version__)

2.18.0-dev20240707


In [2]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

--2024-07-07 17:55:37--  https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
Resolving cdn.freecodecamp.org (cdn.freecodecamp.org)... 172.67.70.149, 104.26.3.33, 104.26.2.33, ...
Connecting to cdn.freecodecamp.org (cdn.freecodecamp.org)|172.67.70.149|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 358233 (350K) [text/tab-separated-values]
Saving to: ‘train-data.tsv’

train-data.tsv      100%[===================>] 349.84K  --.-KB/s    in 0.01s   

2024-07-07 17:55:37 (28.5 MB/s) - ‘train-data.tsv’ saved [358233/358233]

--2024-07-07 17:55:37--  https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv
Resolving cdn.freecodecamp.org (cdn.freecodecamp.org)... 172.67.70.149, 104.26.3.33, 104.26.2.33, ...
Connecting to cdn.freecodecamp.org (cdn.freecodecamp.org)|172.67.70.149|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 118774 (116K) [text/tab-separated-values]
Saving to: ‘valid-data.tsv’

valid-data.tsv      100%[==========

In [3]:
train_dataset = pd.read_csv(train_file_path, sep="\t", names=['type','message'])
train_dataset

,type,message
0,ham,ahhhh...just woken up!had a bad dream about u ...
1,ham,you can never do nothing
2,ham,"now u sound like manky scouse boy steve,like! ..."
3,ham,mum say we wan to go then go... then she can s...
4,ham,never y lei... i v lazy... got wat? dat day ü ...
...,...,...
4174,ham,just woke up. yeesh its late. but i didn't fal...
4175,ham,what do u reckon as need 2 arrange transport i...
4176,spam,free entry into our £250 weekly competition ju...
4177,spam,-pls stop bootydelious (32/f) is inviting you ...


In [4]:
test_dataset = pd.read_table(test_file_path, sep="\t", names=['type','message'])
test_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1392 entries, 0 to 1391
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   type     1392 non-null   object
 1   message  1392 non-null   object
dtypes: object(2)
memory usage: 21.9+ KB


In [5]:
train_dataset['type'].replace(['ham','spam'],[0,1], inplace=True)
test_dataset['type'].replace(['ham','spam'], [0,1], inplace=True)

In [6]:
train_message = train_dataset['message'].values
train_labels = train_dataset['type'].values
train_df = tf.data.Dataset.from_tensor_slices((train_message, train_labels))
train_df.element_spec

(TensorSpec(shape=(), dtype=tf.string, name=None),
 TensorSpec(shape=(), dtype=tf.int64, name=None))

In [7]:
test_message = test_dataset['message'].values
test_labels = test_dataset['type'].values
test_df = tf.data.Dataset.from_tensor_slices((test_message, test_labels))
test_df.element_spec

(TensorSpec(shape=(), dtype=tf.string, name=None),
 TensorSpec(shape=(), dtype=tf.int64, name=None))

In [8]:
BUFFER_SIZE = 1000
BATCH_SIZE = 32
train_df = train_df.shuffle(BUFFER_SIZE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_df = test_df.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [9]:
VOCAB_SIZE = 1000
encoder = tf.keras.layers.TextVectorization(output_mode="int",max_tokens=VOCAB_SIZE)
encoder.adapt(train_df.map(lambda text, label:text))

In [10]:
vocab = np.array(encoder.get_vocabulary())
vocab[:20]

array(['', '[UNK]', 'to', 'i', 'you', 'a', 'the', 'u', 'and', 'in', 'is',
       'me', 'my', 'for', 'your', 'of', 'it', 'call', 'have', 'on'],
      dtype='<U15')

In [11]:
model = tf.keras.Sequential([
    encoder,
    tf.keras.layers.Embedding(len(encoder.get_vocabulary()), 64, mask_zero=True),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64,  return_sequences=True)),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(1)
])

In [12]:
model.compile(loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
              optimizer = tf.keras.optimizers.Adam(1e-4),
              metrics=['accuracy'])

In [13]:
history = model.fit(train_df, epochs=10,
                    validation_data=test_df,
                    validation_steps=30)

Epoch 1/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 33s 147ms/step - accuracy: 0.8685 - loss: 0.6511 - val_accuracy: 0.8604 - val_loss: 0.4753
Epoch 2/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 17s 134ms/step - accuracy: 0.8728 - loss: 0.4183 - val_accuracy: 0.8773 - val_loss: 0.2110
Epoch 3/10


/usr/lib/python3.10/contextlib.py:153: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self.gen.throw(typ, value, traceback)


131/131 ━━━━━━━━━━━━━━━━━━━━ 25s 168ms/step - accuracy: 0.9007 - loss: 0.1750 - val_accuracy: 0.9719 - val_loss: 0.1043
Epoch 4/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 40s 162ms/step - accuracy: 0.9773 - loss: 0.0931 - val_accuracy: 0.9699 - val_loss: 0.0900
Epoch 5/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 40s 156ms/step - accuracy: 0.9818 - loss: 0.0707 - val_accuracy: 0.9812 - val_loss: 0.0667
Epoch 6/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 39s 145ms/step - accuracy: 0.9875 - loss: 0.0586 - val_accuracy: 0.9745 - val_loss: 0.0743
Epoch 7/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 18s 136ms/step - accuracy: 0.9908 - loss: 0.0506 - val_accuracy: 0.9896 - val_loss: 0.0579
Epoch 8/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 18s 134ms/step - accuracy: 0.9901 - loss: 0.0410 - val_accuracy: 0.9769 - val_loss: 0.0641
Epoch 9/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 21s 163ms/step - accuracy: 0.9926 - loss: 0.0342 - val_accuracy: 0.9875 - val_loss: 0.0577
Epoch 10/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 17s 132ms/step - accuracy: 0.9934 - loss: 0.0291 - va

In [23]:
# function to predict messages based on model
# (should return list containing prediction and label, ex. [0.008318834938108921, 'ham'])
def predict_message(pred_text):
  pred_text_array = tf.convert_to_tensor([pred_text])
  prediction = model.predict(pred_text_array)
  print(prediction)
  p = prediction[0]
  if p < 0.5:
    return [p, 'ham']
  else:
    return [p, 'spam']


pred_text = "how are you doing today?"

prediction = predict_message(pred_text)
print(prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
[[-6.2448773]]
[array([-6.2448773], dtype=float32), 'ham']


In [24]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
[[-6.2448773]]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
[[1.2017869]]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
[[-9.397404]]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
[[3.808257]]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
[[5.4444995]]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
[[-9.112193]]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
[[-9.347658]]
You passed the challenge. Great job!
